# 04 — passage-level evidence gate Evidence Gate Debugging Lab

This notebook is a **local, no-API teaching lab**. It constructs retrieved passages deterministically, so you can inspect *answerability logic* without downloading the embedding model. It does **not** measure real FAISS retrieval performance. Run it from the repository root (`jupyter lab`).

In [ ]:
from pathlib import Path
from src.rag.knowledge_base import load_documents
from src.rag.text_chunker import chunk_documents
from src.rag.vector_store import Hit
from src.rag.query_normalizer import deity_names, normalize_query
from src.rag.evidence_gate import concepts, validate_evidence
from src.rag.legacy_evidence_gate import validate_legacy

chunks = {chunk.document_id: chunk for chunk in chunk_documents(load_documents())}
names = deity_names()

def fake_hits(*ids):
    return [Hit(chunks[doc_id], 0.72) for doc_id in ids]

print(f"Documents: {len(chunks)}; named deities: {len(names)}")

## 1. Compare legacy union-of-passages vs passage-level evidence gate support decisions

Notice how separate chunks are no longer treated as one answer.

In [ ]:
cases = [
    ("Why did Demeter grieve when Persephone disappeared?", ("demeter-persephone",)),
    ("What does Hypnos personify?", ("hypnos-sleep",)),
    ("What model of car did Apollo drive?", ("profile-apollo", "profile-helios")),
    ("What smartphone did Zeus use?", ("zeus-sky", "profile-zeus")),
    ("How are apolo and atemis related?", ("apollo-artemis",)),
]
rows = []
for question, ids in cases:
    normalized, corrections = normalize_query(question, names)
    hits = fake_hits(*ids)
    old = validate_legacy(normalized, hits, names)
    new = validate_evidence(normalized, hits, names)
    rows.append({"query": normalized, "legacy union-of-passages": old.status, "passage-level evidence gate": new.status,
                 "coverage": new.coverage, "supporting_doc": new.supporting_document_id})
import pandas as pd
pd.DataFrame(rows)

## 2. Inspect the concepts

A tiny manually authored synonym map helps connect different words, but it is not a learned semantic model.

In [ ]:
for text in ["Why did Demeter grieve when Persephone disappeared?",
             "What does Hypnos personify?",
             "What model of car did Apollo drive?"]:
    print(text, "=>", sorted(concepts(text, names)))
print("Demeter source concepts =>", sorted(concepts(chunks["demeter-persephone"].text, names)))

## 3. An honest failure case

The gate can reject a legitimate paraphrase when our tiny ontology lacks required concepts, or accept a misleading passage. Think about how to collect independent labeled examples rather than tuning all rules to this notebook.

In [ ]:
q = "What plant and bird are traditional Athena symbols?"
result = validate_evidence(q, fake_hits("athena-wisdom"), names)
print(result)
print(chunks["athena-wisdom"].text)

## 4. Real end-to-end evaluation

After dependencies and the index are installed, run this **in a terminal**, not in the notebook:

```powershell
.venv\Scripts\python -m src.rag.evaluate --suite both --compare --output reports/local_retrieval.json
```

The `challenge` prompt set is separately authored but still corpus-derived; results are regression checks, not generalizable accuracy. See `docs/EVIDENCE_VALIDATION_GUIDE.md`.